# Unorganized thoughts and bad calculations

NOTE -
Assuming that any "missing" days are ones in which the balance went below $10. 

I added new rows for the missing days.

Thus, 'active_day' acts as the binary flag for this - 1 indicates an active day, 0 indicates an inactive day in which the balance was below $10.

In [1]:
import pandas as pd

In [2]:
equity_df = pd.read_csv('equity_value_data_v2.csv')
equity_df

,timestamp,close_equity,user_id,date,daily_equity_change
0,2016-08-18T00:00:00Z,1211.6055,0012db34aa7b083f5714e7831195e54d,2016-08-18,NaN
1,2016-08-19T00:00:00Z,1173.5640,0012db34aa7b083f5714e7831195e54d,2016-08-19,-38.0415
2,2016-08-22T00:00:00Z,1253.0597,0012db34aa7b083f5714e7831195e54d,2016-08-22,79.4957
3,2016-08-23T00:00:00Z,1252.9050,0012db34aa7b083f5714e7831195e54d,2016-08-23,-0.1547
4,2016-08-24T00:00:00Z,1262.1360,0012db34aa7b083f5714e7831195e54d,2016-08-24,9.2310
...,...,...,...,...,...
1119153,2017-08-14T00:00:00Z,2156.2400,ffc1e622f3a0b2666f09a6dcb7f27918,2017-08-14,47.9100
1119154,2017-08-15T00:00:00Z,2134.7100,ffc1e622f3a0b2666f09a6dcb7f27918,2017-08-15,-21.5300
1119155,2017-08-16T00:00:00Z,2152.1200,ffc1e622f3a0b2666f09a6dcb7f27918,2017-08-16,17.4100
1119156,2017-08-17T00:00:00Z,2042.2800,ffc1e622f3a0b2666f09a6dcb7f27918,2017-08-17,-109.8400


In [3]:
# Function to create full date range for one user
def complete_user_dates(user_df):
    full_dates = pd.DataFrame({'date': pd.date_range(user_df['date'].min(), user_df['date'].max())})
    full_dates['user_id'] = user_df['user_id'].iloc[0]
    return full_dates

complete_dates_df = (
    equity_df.groupby('user_id')
    .apply(complete_user_dates)
    .reset_index(drop=True)
)

equity_df['date'] = pd.to_datetime(equity_df['date'])
complete_dates_df['date'] = pd.to_datetime(complete_dates_df['date'])

/var/folders/cf/4zrkk0lx1nbdx9tpklpk4jc00000gn/T/ipykernel_45807/1414377222.py:9: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(complete_user_dates)


In [4]:
# Merge to bring in actual trades
full_equity_df = complete_dates_df.merge(
    equity_df[['user_id', 'date', 'close_equity', 'timestamp']],
    on=['user_id', 'date'],
    how='left'
)

# Sort new DataFrame
full_equity_df = full_equity_df.sort_values(['user_id', 'date'])

# Add binary flag for active/inactive days
full_equity_df['active_day'] = full_equity_df['timestamp'].notna().astype(int)

# HOW TO TREAT CLOSE_EQUITY ON INACTIVE DAYS?
# Fill null close_equity with $9
full_equity_df['close_equity'] = full_equity_df['close_equity'].fillna(9)

# DAILY CHANGE? DOES THIS MAKE SENSE IF I AM LEAVING INACTIVE DAYS NULL?
# Daily change
# full_equity_df['daily_equity_change'] = full_equity_df.groupby('user_id')['close_equity'].diff()

In [5]:
# Drop timestamp as it no longer has useful information after date was split off.
full_equity_df.drop('timestamp', axis=1, inplace=True)

In [6]:
full_equity_df = full_equity_df.sort_values(['user_id', 'date'])

# Create a group that increments every time we hit an active day
grp = (
    full_equity_df['active_day']
    .eq(1)
    .groupby(full_equity_df['user_id'])
    .cumsum()
)

# Count days since last active day
full_equity_df['days_since_last_active'] = (
    full_equity_df
    .groupby(['user_id', grp])
    .cumcount()
)

In [7]:
full_equity_df['streak_at_least_28'] = (full_equity_df['days_since_last_active'] >= 28).astype(int)

In [8]:
full_equity_df

,date,user_id,close_equity,active_day,days_since_last_active,streak_at_least_28
0,2016-08-18,0012db34aa7b083f5714e7831195e54d,1211.6055,1,0,0
1,2016-08-19,0012db34aa7b083f5714e7831195e54d,1173.5640,1,0,0
2,2016-08-20,0012db34aa7b083f5714e7831195e54d,9.0000,0,1,0
3,2016-08-21,0012db34aa7b083f5714e7831195e54d,9.0000,0,2,0
4,2016-08-22,0012db34aa7b083f5714e7831195e54d,1253.0597,1,0,0
...,...,...,...,...,...,...
1647990,2017-08-14,ffc1e622f3a0b2666f09a6dcb7f27918,2156.2400,1,0,0
1647991,2017-08-15,ffc1e622f3a0b2666f09a6dcb7f27918,2134.7100,1,0,0
1647992,2017-08-16,ffc1e622f3a0b2666f09a6dcb7f27918,2152.1200,1,0,0
1647993,2017-08-17,ffc1e622f3a0b2666f09a6dcb7f27918,2042.2800,1,0,0


In [9]:
full_equity_df['streak_at_least_28'].value_counts()

streak_at_least_28
0    1631744
1      16251
Name: count, dtype: int64

In [10]:
full_equity_df.nlargest(10, 'days_since_last_active')

,date,user_id,close_equity,active_day,days_since_last_active,streak_at_least_28
809736,2017-07-19,7f9dee557e08cd3748fe9f84bb3afc8d,9.0,0,334,1
809735,2017-07-18,7f9dee557e08cd3748fe9f84bb3afc8d,9.0,0,333,1
809734,2017-07-17,7f9dee557e08cd3748fe9f84bb3afc8d,9.0,0,332,1
809733,2017-07-16,7f9dee557e08cd3748fe9f84bb3afc8d,9.0,0,331,1
809732,2017-07-15,7f9dee557e08cd3748fe9f84bb3afc8d,9.0,0,330,1
809731,2017-07-14,7f9dee557e08cd3748fe9f84bb3afc8d,9.0,0,329,1
809730,2017-07-13,7f9dee557e08cd3748fe9f84bb3afc8d,9.0,0,328,1
809729,2017-07-12,7f9dee557e08cd3748fe9f84bb3afc8d,9.0,0,327,1
809728,2017-07-11,7f9dee557e08cd3748fe9f84bb3afc8d,9.0,0,326,1
809727,2017-07-10,7f9dee557e08cd3748fe9f84bb3afc8d,9.0,0,325,1


# Features

In [11]:
features_df = pd.read_csv('features_data.csv')
features_df

,risk_tolerance,investment_experience,liquidity_needs,platform,time_spent,instrument_type_first_traded,first_deposit_amount,time_horizon,user_id
0,high_risk_tolerance,limited_investment_exp,very_important_liq_need,Android,33.129417,stock,40.0,med_time_horizon,895044c23edc821881e87da749c01034
1,med_risk_tolerance,limited_investment_exp,very_important_liq_need,Android,16.573517,stock,200.0,short_time_horizon,458b1d95441ced242949deefe8e4b638
2,med_risk_tolerance,limited_investment_exp,very_important_liq_need,iOS,10.008367,stock,25.0,long_time_horizon,c7936f653d293479e034865db9bb932f
3,med_risk_tolerance,limited_investment_exp,very_important_liq_need,Android,1.031633,stock,100.0,short_time_horizon,b255d4bd6c9ba194d3a350b3e76c6393
4,high_risk_tolerance,limited_investment_exp,very_important_liq_need,Android,8.187250,stock,20.0,long_time_horizon,4a168225e89375b8de605cbc0977ae91
...,...,...,...,...,...,...,...,...,...
5579,high_risk_tolerance,limited_investment_exp,very_important_liq_need,Android,8.339283,stock,300.0,long_time_horizon,03880c726d8a4e5db006afe4119ad974
5580,med_risk_tolerance,limited_investment_exp,somewhat_important_liq_need,iOS,7.241383,stock,100.0,short_time_horizon,ae8315109657f44852b24c6bca4decd6
5581,med_risk_tolerance,no_investment_exp,very_important_liq_need,both,22.967167,stock,50.0,short_time_horizon,f29c174989f9737058fe808fcf264135
5582,med_risk_tolerance,limited_investment_exp,somewhat_important_liq_need,iOS,10.338417,stock,100.0,long_time_horizon,24843497d1de88b2e7233f694436cb3a


In [12]:
# Create flag for users who ever dropped below 10
low_equity_flag = full_equity_df.groupby('user_id')['active_day'].apply(lambda x: (x == 0).any()).astype(int)
# Merge / map into features_df
features_df['ever_below_10'] = features_df['user_id'].map(low_equity_flag)

# Create flag for users with 28 day inactive streak
inactive_streak = full_equity_df.groupby('user_id')['days_since_last_active'].apply(lambda x: (x >= 28).any()).astype(int)
features_df['ever_inactive_28_days'] = features_df['user_id'].map(low_equity_flag)

features_df

,risk_tolerance,investment_experience,liquidity_needs,platform,time_spent,instrument_type_first_traded,first_deposit_amount,time_horizon,user_id,ever_below_10,ever_inactive_28_days
0,high_risk_tolerance,limited_investment_exp,very_important_liq_need,Android,33.129417,stock,40.0,med_time_horizon,895044c23edc821881e87da749c01034,1,1
1,med_risk_tolerance,limited_investment_exp,very_important_liq_need,Android,16.573517,stock,200.0,short_time_horizon,458b1d95441ced242949deefe8e4b638,1,1
2,med_risk_tolerance,limited_investment_exp,very_important_liq_need,iOS,10.008367,stock,25.0,long_time_horizon,c7936f653d293479e034865db9bb932f,1,1
3,med_risk_tolerance,limited_investment_exp,very_important_liq_need,Android,1.031633,stock,100.0,short_time_horizon,b255d4bd6c9ba194d3a350b3e76c6393,1,1
4,high_risk_tolerance,limited_investment_exp,very_important_liq_need,Android,8.187250,stock,20.0,long_time_horizon,4a168225e89375b8de605cbc0977ae91,1,1
...,...,...,...,...,...,...,...,...,...,...,...
5579,high_risk_tolerance,limited_investment_exp,very_important_liq_need,Android,8.339283,stock,300.0,long_time_horizon,03880c726d8a4e5db006afe4119ad974,1,1
5580,med_risk_tolerance,limited_investment_exp,somewhat_important_liq_need,iOS,7.241383,stock,100.0,short_time_horizon,ae8315109657f44852b24c6bca4decd6,1,1
5581,med_risk_tolerance,no_investment_exp,very_important_liq_need,both,22.967167,stock,50.0,short_time_horizon,f29c174989f9737058fe808fcf264135,1,1
5582,med_risk_tolerance,limited_investment_exp,somewhat_important_liq_need,iOS,10.338417,stock,100.0,long_time_horizon,24843497d1de88b2e7233f694436cb3a,1,1


In [13]:
features_df['ever_below_10'].value_counts()

ever_below_10
1    5560
0      24
Name: count, dtype: int64

In [14]:
features_df['ever_inactive_28_days'].value_counts()

ever_inactive_28_days
1    5560
0      24
Name: count, dtype: int64

# New dataframe for inactive streaks.

In [15]:
inactive_streaks_df = pd.DataFrame()
inactive_streaks_df['user_id'] = full_equity_df['user_id']
inactive_streaks_df['days_since_last_active'] = full_equity_df['days_since_last_active']
inactive_streaks_df

,user_id,days_since_last_active
0,0012db34aa7b083f5714e7831195e54d,0
1,0012db34aa7b083f5714e7831195e54d,0
2,0012db34aa7b083f5714e7831195e54d,1
3,0012db34aa7b083f5714e7831195e54d,2
4,0012db34aa7b083f5714e7831195e54d,0
...,...,...
1647990,ffc1e622f3a0b2666f09a6dcb7f27918,0
1647991,ffc1e622f3a0b2666f09a6dcb7f27918,0
1647992,ffc1e622f3a0b2666f09a6dcb7f27918,0
1647993,ffc1e622f3a0b2666f09a6dcb7f27918,0


In [16]:
inactive_streaks_df = full_equity_df.copy()
inactive_streaks_df = inactive_streaks_df[inactive_streaks_df['streak_at_least_28'] == 1]
inactive_streaks_df

,date,user_id,close_equity,active_day,days_since_last_active,streak_at_least_28
1543,2017-03-27,00440034cc4152bfb01b30f5c381c4e3,9.0,0,28,1
1544,2017-03-28,00440034cc4152bfb01b30f5c381c4e3,9.0,0,29,1
1545,2017-03-29,00440034cc4152bfb01b30f5c381c4e3,9.0,0,30,1
1546,2017-03-30,00440034cc4152bfb01b30f5c381c4e3,9.0,0,31,1
1547,2017-03-31,00440034cc4152bfb01b30f5c381c4e3,9.0,0,32,1
...,...,...,...,...,...,...
1646447,2017-08-09,ff98cbc9633a9255a711d02ebc1e8294,9.0,0,68,1
1646448,2017-08-10,ff98cbc9633a9255a711d02ebc1e8294,9.0,0,69,1
1646449,2017-08-11,ff98cbc9633a9255a711d02ebc1e8294,9.0,0,70,1
1646450,2017-08-12,ff98cbc9633a9255a711d02ebc1e8294,9.0,0,71,1


In [17]:
# Ensure proper ordering
inactive_streaks_df = inactive_streaks_df.sort_values(['user_id', 'date'])

# Reset streak when it ends
reset = (
    inactive_streaks_df['active_day']
    .eq(1)
    .groupby(inactive_streaks_df['user_id'])
    .cumsum()
)

# Assign streak_id
inactive_streaks_df['streak_id'] = (
    inactive_streaks_df
    .groupby(['user_id', reset])
    .cumcount()
)

In [18]:
inactive_streaks_df

,date,user_id,close_equity,active_day,days_since_last_active,streak_at_least_28,streak_id
1543,2017-03-27,00440034cc4152bfb01b30f5c381c4e3,9.0,0,28,1,0
1544,2017-03-28,00440034cc4152bfb01b30f5c381c4e3,9.0,0,29,1,1
1545,2017-03-29,00440034cc4152bfb01b30f5c381c4e3,9.0,0,30,1,2
1546,2017-03-30,00440034cc4152bfb01b30f5c381c4e3,9.0,0,31,1,3
1547,2017-03-31,00440034cc4152bfb01b30f5c381c4e3,9.0,0,32,1,4
...,...,...,...,...,...,...,...
1646447,2017-08-09,ff98cbc9633a9255a711d02ebc1e8294,9.0,0,68,1,40
1646448,2017-08-10,ff98cbc9633a9255a711d02ebc1e8294,9.0,0,69,1,41
1646449,2017-08-11,ff98cbc9633a9255a711d02ebc1e8294,9.0,0,70,1,42
1646450,2017-08-12,ff98cbc9633a9255a711d02ebc1e8294,9.0,0,71,1,43


In [19]:
inactive_streaks_df['first_day_of_streak'] = (
    inactive_streaks_df
    .groupby(['user_id', 'streak_id'])['date']
    .transform('min')
)

In [20]:
inactive_streaks_df

,date,user_id,close_equity,active_day,days_since_last_active,streak_at_least_28,streak_id,first_day_of_streak
1543,2017-03-27,00440034cc4152bfb01b30f5c381c4e3,9.0,0,28,1,0,2017-03-27
1544,2017-03-28,00440034cc4152bfb01b30f5c381c4e3,9.0,0,29,1,1,2017-03-28
1545,2017-03-29,00440034cc4152bfb01b30f5c381c4e3,9.0,0,30,1,2,2017-03-29
1546,2017-03-30,00440034cc4152bfb01b30f5c381c4e3,9.0,0,31,1,3,2017-03-30
1547,2017-03-31,00440034cc4152bfb01b30f5c381c4e3,9.0,0,32,1,4,2017-03-31
...,...,...,...,...,...,...,...,...
1646447,2017-08-09,ff98cbc9633a9255a711d02ebc1e8294,9.0,0,68,1,40,2017-08-09
1646448,2017-08-10,ff98cbc9633a9255a711d02ebc1e8294,9.0,0,69,1,41,2017-08-10
1646449,2017-08-11,ff98cbc9633a9255a711d02ebc1e8294,9.0,0,70,1,42,2017-08-11
1646450,2017-08-12,ff98cbc9633a9255a711d02ebc1e8294,9.0,0,71,1,43,2017-08-12


In [21]:
inactive_streaks_df['user_id'].nunique()

275

In [22]:
inactive_streaks_df['streak_id'].nunique()

307

In [23]:
inactive_streaks_df['streak_id_2'] = (
    inactive_streaks_df
    .groupby('user_id')['active_day']
    .apply(
        lambda x: ((x == 0) & (x.shift(fill_value=1) == 1)).cumsum()
    )
    .reset_index(level=0, drop=True)
)

In [24]:
inactive_streaks_df

,date,user_id,close_equity,active_day,days_since_last_active,streak_at_least_28,streak_id,first_day_of_streak,streak_id_2
1543,2017-03-27,00440034cc4152bfb01b30f5c381c4e3,9.0,0,28,1,0,2017-03-27,1
1544,2017-03-28,00440034cc4152bfb01b30f5c381c4e3,9.0,0,29,1,1,2017-03-28,1
1545,2017-03-29,00440034cc4152bfb01b30f5c381c4e3,9.0,0,30,1,2,2017-03-29,1
1546,2017-03-30,00440034cc4152bfb01b30f5c381c4e3,9.0,0,31,1,3,2017-03-30,1
1547,2017-03-31,00440034cc4152bfb01b30f5c381c4e3,9.0,0,32,1,4,2017-03-31,1
...,...,...,...,...,...,...,...,...,...
1646447,2017-08-09,ff98cbc9633a9255a711d02ebc1e8294,9.0,0,68,1,40,2017-08-09,1
1646448,2017-08-10,ff98cbc9633a9255a711d02ebc1e8294,9.0,0,69,1,41,2017-08-10,1
1646449,2017-08-11,ff98cbc9633a9255a711d02ebc1e8294,9.0,0,70,1,42,2017-08-11,1
1646450,2017-08-12,ff98cbc9633a9255a711d02ebc1e8294,9.0,0,71,1,43,2017-08-12,1


In [46]:
inactive_streaks_df.groupby(['user_id','streak_id']).size().value_counts()


1    16251
Name: count, dtype: int64

In [51]:
# Ensure 'date' is datetime
inactive_streaks_df['date'] = pd.to_datetime(inactive_streaks_df['date'])

# For each streak, find the index of the row with the earliest date
idx_earliest = inactive_streaks_df.groupby('streak_id')['date'].idxmin()

# Select those rows
streaks_df = inactive_streaks_df.loc[idx_earliest].copy().reset_index(drop=True)


In [52]:
streaks_df

,date,user_id,close_equity,active_day,days_since_last_active,streak_at_least_28,streak_id,first_day_of_streak,streak_id_2
0,2016-09-15,c0eb9d1466bd8e338cefe499882ad91b,9.0,0,28,1,0,2016-09-15,1
1,2016-09-16,c0eb9d1466bd8e338cefe499882ad91b,9.0,0,29,1,1,2016-09-16,1
2,2016-09-17,c0eb9d1466bd8e338cefe499882ad91b,9.0,0,30,1,2,2016-09-17,1
3,2016-09-18,c0eb9d1466bd8e338cefe499882ad91b,9.0,0,31,1,3,2016-09-18,1
4,2016-09-19,c0eb9d1466bd8e338cefe499882ad91b,9.0,0,32,1,4,2016-09-19,1
...,...,...,...,...,...,...,...,...,...
302,2017-07-15,7f9dee557e08cd3748fe9f84bb3afc8d,9.0,0,330,1,302,2017-07-15,1
303,2017-07-16,7f9dee557e08cd3748fe9f84bb3afc8d,9.0,0,331,1,303,2017-07-16,1
304,2017-07-17,7f9dee557e08cd3748fe9f84bb3afc8d,9.0,0,332,1,304,2017-07-17,1
305,2017-07-18,7f9dee557e08cd3748fe9f84bb3afc8d,9.0,0,333,1,305,2017-07-18,1


In [55]:
streaks_df = streaks_df.sort_values(['user_id', 'date'])

# Merge the new columns back into the original dataframe

In [60]:
# Ensure date is datetime
full_equity_df['date'] = pd.to_datetime(full_equity_df['date'])
streaks_df['first_day_of_streak'] = pd.to_datetime(streaks_df['first_day_of_streak'])


In [61]:
def equity_before_streak(user_id, streak_start):
    # All days for this user before the streak
    user_days = full_equity_df[
        (full_equity_df['user_id'] == user_id) &
        (full_equity_df['date'] < streak_start)
    ]
    return int((user_days['close_equity'] >= 10).any())


In [62]:
streaks_df['equity_above_10_before_streak'] = streaks_df.apply(
    lambda row: equity_before_streak(row['user_id'], row['first_day_of_streak']),
    axis=1
)


In [63]:
streaks_df

,date,user_id,close_equity,active_day,days_since_last_active,streak_at_least_28,streak_id,first_day_of_streak,streak_id_2,ever_above_10,equity_above_10_before_streak
130,2017-01-24,7f9dee557e08cd3748fe9f84bb3afc8d,9.0,0,158,1,130,2017-01-24,1,1,1
131,2017-01-25,7f9dee557e08cd3748fe9f84bb3afc8d,9.0,0,159,1,131,2017-01-25,1,1,1
132,2017-01-26,7f9dee557e08cd3748fe9f84bb3afc8d,9.0,0,160,1,132,2017-01-26,1,1,1
133,2017-01-27,7f9dee557e08cd3748fe9f84bb3afc8d,9.0,0,161,1,133,2017-01-27,1,1,1
134,2017-01-28,7f9dee557e08cd3748fe9f84bb3afc8d,9.0,0,162,1,134,2017-01-28,1,1,1
...,...,...,...,...,...,...,...,...,...,...,...
125,2017-01-18,fbc5223e0b38db36c9b3439326aae416,9.0,0,153,1,125,2017-01-18,1,1,1
126,2017-01-19,fbc5223e0b38db36c9b3439326aae416,9.0,0,154,1,126,2017-01-19,1,1,1
127,2017-01-20,fbc5223e0b38db36c9b3439326aae416,9.0,0,155,1,127,2017-01-20,1,1,1
128,2017-01-21,fbc5223e0b38db36c9b3439326aae416,9.0,0,156,1,128,2017-01-21,1,1,1
